# 🎮 AIOS Quant RL Training (Stable-Baselines3)

Обучение Reinforcement Learning торгового агента в симулированной биржевой среде.

**T4 GPU**.

Агент решает, какую долю капитала держать в активе, максимизируя доходность и минимизируя просадки. Используем PPO из Stable-Baselines3 на среде, построенной по ценам из ccxt.

In [ ]:
!pip install -q ccxt pandas numpy gymnasium stable-baselines3
import ccxt, pandas as pd, numpy as np
print('✅ Зависимости установлены')

In [ ]:
# === ЯЧЕЙКА 2: Загрузка цен ===
cl = ccxt.binance()
cl.load_markets()
ohlcv = cl.fetch_ohlcv('BTC/USDT', '1h', limit=2000)
df = pd.DataFrame(ohlcv, columns=['ts','open','high','low','close','volume'])
df['returns'] = df['close'].pct_change().fillna(0)
df['momentum'] = df['close'].pct_change(12).fillna(0)
print('✅ Цены:', df.shape)

In [ ]:
# === ЯЧЕЙКА 3: RL-среда (gymnasium) ===
import gymnasium as gym
from gymnasium import spaces

class TradingEnv(gym.Env):
    def __init__(self, df, window=10):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.window = window
        self.action_space = spaces.Discrete(3)  # 0=полный выход, 1=50%, 2=100% в активе
        self.observation_space = spaces.Box(-np.inf, np.inf, (window*2,), dtype=np.float32)
        self.i = window
    def _obs(self):
        w = self.df['returns'].values[self.i-self.window:self.i]
        m = self.df['momentum'].values[self.i-self.window:self.i]
        return np.concatenate([w, m]).astype(np.float32)
    def reset(self, *, seed=None, options=None):
        self.i = self.window
        self.position = 0
        return self._obs(), {}
    def step(self, action):
        pos = action / 2.0
        r = self.df['returns'].values[self.i]
        reward = pos * r * 100  # доходность позиции (x100 для масштаба)
        self.position = pos
        self.i += 1
        done = self.i >= len(self.df) - 1
        return self._obs() if not done else self._obs(), float(reward), done, False, {}

env = TradingEnv(df)
print('✅ Среда создана')

In [ ]:
# === ЯЧЕЙКА 4: Обучение PPO ===
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

vec = DummyVecEnv([lambda: TradingEnv(df)])
model = PPO('MlpPolicy', vec, verbose=0, learning_rate=1e-3, n_steps=512)
model.learn(total_timesteps=20000)
print('✅ PPO обучен')

In [ ]:
# === ЯЧЕЙКА 5: Валидация агента + сохранение + выгрузка на Google Диск ===
import os, shutil
val = DummyVecEnv([lambda: TradingEnv(df.iloc[len(df)//2:].reset_index(drop=True))])
obs = val.reset()
total = 0
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = val.step(action)
    total += reward
print(fИтоговая доходность агента на валидации: {total:.2f}%)
os.makedirs(/content/models, exist_ok=True)
model.save(/content/models/ppo_trader.zip)
print(✅ Агент сохранён: /content/models/ppo_trader.zip)

# Выгрузка на Google Диск (если смонтирован или можно смонтировать)
DRIVE_OK = False
try:
    from google.colab import drive
    if not os.path.isdir(/content/drive/MyDrive):
        drive.mount(/content/drive)
    dst = /content/drive/MyDrive/AIOS_colab_models
    os.makedirs(dst, exist_ok=True)
    shutil.copy2(/content/models/ppo_trader.zip, os.path.join(dst, ppo_trader.zip))
    DRIVE_OK = True
    print(DRIVE_COPY_DONE, os.path.exists(os.path.join(dst, ppo_trader.zip)))
except Exception as e:
    print(DRIVE_COPY_ERR, repr(e)[:150])

print(RL_DONE, DRIVE_OK if DRIVE_OK else LOCAL_ONLY)
